load_documents
split_documents
create_embed_model
build_vector_store
load_vector_store
retrieve_documents
format_context
generate_answer

In [2]:
from pathlib import Path
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.rand(2, 2))

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

from src import config


c:\Users\thorp\Documents\GitHub\rag-project\.venv\Scripts\python.exe
2.13.0+cpu
tensor([[0.5352, 0.9745],
        [0.4081, 0.0692]])


In [5]:
import bs4
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_classic import hub
from langchain_core.document_loaders import BaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough



# Load Documents method
def load_documents(file_path: str) -> list[Document]:
    """
    Loads Every PDF in a directory as page-level LangChain documents
    """
    documents = []

    for pdf_path in Path(file_path).glob("*pdf"):
        reader = PdfReader(pdf_path)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text()

            if text and text.strip():
                documents.append(
                    Document(
                        page_content=text.strip(),
                        metadata={
                            "source":pdf_path.name,
                            "page": page_number
                        }
                    )
                )
    if not documents:
        raise ValueError(f"No readable PDFs found in {file_path}")

    return documents

# Loads the F1 Regulations
docs = load_documents(config.RAW_DATA_DIR)
print(f"Loaded {len(docs)} pages")
print(docs[0].metadata)
print(docs[0].page_content[:500])


c:\Users\thorp\Documents\GitHub\rag-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 215 pages
{'source': 'general_provisions.pdf', 'page': 1}
SECTION A: GENERAL REGULATORY PROVISIONS 
 
 
 
0 
 
A 
 
  
A1 
2026 Formula 1: General Regulatory Provisions 
©2026 Fédération Internationale de l’Automobile 
25 June 2026 
Issue 03 
SECTION A: GENERAL REGULATORY PROVISIONS 
 
Version:   Issue 03 
Status:    PUBLISHED 
Date:    25/06/2026 
WMSC approval date:  23/06/2026 
 
CONVENTION: 
Black Text: Text unchanged from 2026 FIA F1 Regulations – Section A [General Regulatory Provisions] – Iss 
02 
Pink Text: Changes relative to 2026 FIA F1 Regul


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split method
def split_documents(
        documents,
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP
    ):
    """
    Splits documents into a split list of documents based on chunk parameters
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    return text_splitter.split_documents(documents)

# Split the F1 Regulations
splits = split_documents(docs)

print(f"Created {len(splits)} chunks")
print(splits[0].metadata)
print(splits[0].page_content[:500])

Created 931 chunks
{'source': 'general_provisions.pdf', 'page': 1}
SECTION A: GENERAL REGULATORY PROVISIONS 
 
 
 
0 
 
A 
 
  
A1 
2026 Formula 1: General Regulatory Provisions 
©2026 Fédération Internationale de l’Automobile 
25 June 2026 
Issue 03 
SECTION A: GENERAL REGULATORY PROVISIONS 
 
Version:   Issue 03 
Status:    PUBLISHED 
Date:    25/06/2026 
WMSC approval date:  23/06/2026 
 
CONVENTION: 
Black Text: Text unchanged from 2026 FIA F1 Regulations – Section A [General Regulatory Provisions] – Iss 
02 
Pink Text: Changes relative to 2026 FIA F1 Regul


In [8]:
# Embed and Retrieve

from src import config
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="f1_regulations",
    persist_directory=str(config.VECTOR_STORE_DIR),
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": config.TOP_K}
)


c:\Users\thorp\Documents\GitHub\rag-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\thorp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14663.91it/s]


In [ ]:
"""
# Retriever Test

question = "What happens when a race is suspended?"

results = retriever.invoke(question)

for result in results:
    print(result.metadata)
    print(result.page_content[:700])
    print("-" * 80)
"""

{'page': 24, 'source': 'sporting_regulations.pdf'}
SECTION B: SPORTING REGULATIONS 
 
 
0 
 
B 
 
 
B24 
2026 Formula 1: Sporting Regulations 
©2026 Fédération Internationale de l’Automobile 
25 June 2026 
Issue 07  
b. Should the Race be suspended (Article B5.14.2), the duration of the suspension will be added 
to this two (2) hour period up to a maximum total Race duration of three (3) hours, and the 
leader will be shown the end -of-session signal when they cross the Line at the end of the lap 
following the lap during which the sum of these periods ended, provided this does not result in 
the scheduled number of laps being exceeded.  
If the formation lap for the Race is started behind the Safety Car (Article B5.10), the maximum 
total R
--------------------------------------------------------------------------------
{'page': 14, 'source': 'sporting_regulations.pdf'}
j. Suspension from the driver’s next Competition. 
B1.9.6 Procedure(s) for Serving a Penalty 
Should the stewards im